# 02_Modeling_Evaluation: Huấn luyện và Đánh giá Mô hình Học máy

**Học phần:** Học máy (608807) – Lớp 69CS1  
**Đề tài 3:** Định giá Bất động sản Ames Housing  
**Người thực hiện:** Dương Hữu Hinh | MSSV: 0367869 (Phần việc: Pipeline & Thử nghiệm mô hình)

## 1. Đọc dữ liệu và chia train/test

Tập test (20%, `random_state=42`) đã được chia một lần và lưu Id ở `data/processed/test_ids.csv`.
Mọi phân tích và bước fit chỉ dùng **tập train**; tập test chỉ dùng để đánh giá cuối, tránh rò rỉ dữ liệu.

In [1]:
import sys
from pathlib import Path

# Thêm 03_SourceCode/ vào sys.path để import được thư mục src/
sys.path.append(str(Path.cwd().parent))

from src.data import RANDOM_STATE, load_split

X_train, X_test, y_train, y_test = load_split()
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (1168, 79) | Test: (292, 79)


## 2. Kiến trúc Pipeline

### Sơ đồ kiến trúc Pipeline 5 bước:
```text
Dữ liệu thô (X_train)
      │
      ▼
1. MissingHandler (Học median LotFrontage theo khu & mode từ train fold)
      │
      ▼
2. OrdinalMapper (Mã hóa thứ bậc các cột chất lượng: ExterQual, KitchenQual... 0-5)
      │
      ▼
3. FeatureAdder (Tạo đặc trưng: TotalSF, HouseAge, RemodAge, TotalBath, TotalPorchSF, flags...)
      │
      ▼
4. ColumnTransformer:
   ├── Nhánh số: SimpleImputer(median) ──► StandardScaler()
   └── Nhánh phân loại: SimpleImputer("None") ──► OneHotEncoder(min_frequency=0.01)
      │
      ▼
5. Mô hình Hồi quy (Dummy, Ridge, Lasso, KNN, DecisionTree, RandomForest, HistGB)
   (Được bọc trong TransformedTargetRegressor với func=np.log1p, inverse_func=np.expm1)
```

**Nguyên tắc chống rò rỉ dữ liệu (No Data Leakage):**
- Mọi bước học thông số từ dữ liệu (median theo phân vùng `Neighborhood`, giá trị mode, giá trị trung bình/độ lệch chuẩn của `StandardScaler`, danh mục của `OneHotEncoder`) đều nằm hoàn toàn bên trong các phương thức `fit()` của Pipeline.
- Khi kiểm định chéo K-Fold Cross Validation, mỗi fold huấn luyện sẽ fit lại độc lập toàn bộ các bước tiền xử lý này, không để bất kỳ thông tin nào của fold validation rò rỉ ra ngoài.
- **Xử lý ngoại lai nằm ngoài Pipeline:** Bước loại bỏ ngoại lai `remove_outliers` (bỏ nhà có `GrLivArea > 4000`) được thực hiện trước khi đưa vào Pipeline và **chỉ gọi trên tập train**, vì transformer của Scikit-learn chỉ có thể biến đổi cột/giá trị chứ không thể tự xóa bớt số dòng của ma trận dữ liệu. Tập test giữ nguyên để phản ánh dữ liệu thực tế.

In [2]:
from src.data import remove_outliers
from src.pipeline import build_pipeline, get_models

# Bỏ ngoại lai chỉ trên train (ngoài Pipeline vì transformer không xoá được dòng)
n_before = len(X_train)
X_train, y_train = remove_outliers(X_train, y_train)
print(f"Đã loại {n_before - len(X_train)} ngoại lai (GrLivArea > 4000).")
print(f"Train còn: {X_train.shape}")

build_pipeline(get_models()["Ridge"])  # hiển thị sơ đồ Pipeline

Đã loại 3 ngoại lai (GrLivArea > 4000).
Train còn: (1165, 79)


TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('missing', MissingHandler()),
                                                   ('ordinal', OrdinalMapper()),
                                                   ('fe', FeatureAdder()),
                                                   ('prep',
                                                    ColumnTransformer(transformers=[('num',
                                                                                     Pipeline(steps=[('imputer',
                                                                                                      SimpleImputer(strategy='median')),
                                                                                                     ('scaler',
                                                                                                      StandardScaler())]),
                                            

## 3. So sánh 7 mô hình bằng 5-Fold CV (chỉ trên train)

Thực hiện đánh giá K-Fold Cross Validation (5 folds, `shuffle=True`, `random_state=42`) trên tập train sạch (1.165 dòng) qua 4 độ đo chuẩn:
- **RMSLE** (Root Mean Squared Log Error): Độ đo chính của cuộc thi Kaggle Ames Housing.
- **$R^2$ Score**: Hệ số xác định, đo lường tỉ lệ phương sai giá nhà được giải thích.
- **RMSE** (Root Mean Squared Error): Sai số theo đơn vị USD.
- **MAE** (Mean Absolute Error): Sai số tuyệt đối trung bình theo đơn vị USD.

In [3]:
import pandas as pd
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "rmsle": "neg_root_mean_squared_log_error",
}

rows = {}
for name, model in get_models().items():
    res = cross_validate(
        build_pipeline(model), X_train, y_train, cv=cv, scoring=scoring
    )
    rows[name] = {
        "RMSE": -res["test_rmse"].mean(),
        "MAE": -res["test_mae"].mean(),
        "R2": res["test_r2"].mean(),
        "RMSLE": -res["test_rmsle"].mean(),
        "RMSLE_std": res["test_rmsle"].std(),
    }

bang_cv = pd.DataFrame(rows).T.sort_values("RMSLE")
bang_cv.round(4)

,RMSE,MAE,R2,RMSLE,RMSLE_std
Lasso,20081.8798,13655.5201,0.9291,0.1106,0.0156
Ridge,20549.4447,13928.9134,0.9257,0.1138,0.0134
HistGB,22955.9375,15042.7955,0.9067,0.1231,0.0157
RandomForest,25119.1512,16530.5505,0.8885,0.1330,0.0136
KNN,32346.5560,20658.7854,0.8152,0.1585,0.0100
DecisionTree,35722.0622,24464.4955,0.7754,0.1981,0.0063
Dummy,76722.8524,54266.3627,-0.0340,0.3887,0.0057


### Nhận xét kết quả 5-Fold Cross Validation:
1. **Mô hình tốt nhất:**
   - **Lasso Regression** đạt hiệu quả cao nhất với **RMSLE = 0.1106** ($\pm 0.0156$), $R^2 = 0.9291$, RMSE = $20,081.88, MAE = $13,655.52.
   - **Ridge Regression** bám sát rất gần với **RMSLE = 0.1138** ($\pm 0.0134$), $R^2 = 0.9257$, MAE = $13,928.91.
2. **So sánh Tuyến tính (Lasso/Ridge) với Cây và Ensemble:**
   - Các mô hình tuyến tính có điều chuẩn (Lasso, Ridge) cho kết quả vượt trội hơn so với các mô hình cây và ensemble (HistGB: RMSLE 0.1231, RandomForest: 0.1330, DecisionTree: 0.1981).
   - Lý do: Biến mục tiêu `SalePrice` sau khi biến đổi `log1p` có phân phối xấp xỉ chuẩn, kết hợp với các đặc trưng diện tích (`TotalSF`, `GrLivArea`) và chất lượng có mối quan hệ tuyến tính rất mạnh với log-giá nhà. Điều chuẩn L1/L2 giúp kiểm soát tốt đa cộng tuyến mà không bị overfitting như các mô hình phi tuyến.
3. **Mức cải thiện so với Baseline (DummyRegressor):**
   - DummyRegressor chỉ đạt RMSLE = 0.3887, $R^2 = -0.0340$, MAE = $54,266.36.
   - Lasso giúp giảm sai số tuyệt đối MAE từ $54,266 xuống còn $13,655 (giảm gần 75% sai số), khẳng định Pipeline và các mô hình học máy đã học được tri thức rất tốt từ dữ liệu.
4. **Độ ổn định giữa các fold:**
   - Độ lệch chuẩn RMSLE giữa 5 fold rất thấp (`RMSLE_std` dao động từ 0.0057 đến 0.0157), chứng tỏ mô hình có tính tổng quát hóa cao, không bị phụ thuộc vào một phân vùng dữ liệu cụ thể.

### So sánh phương thức mã hóa: Ordinal Encoding vs. One-Hot Encoding

Thực nghiệm đối chứng: Chạy lại 5-Fold CV khi tắt `OrdinalMapper` (`ordinal=False`), cho phép toàn bộ 12 cột chất lượng đi qua One-Hot Encoding để so sánh trong báo cáo:

In [4]:
rows_no_ord = {}
for name, model in get_models().items():
    res = cross_validate(
        build_pipeline(model, ordinal=False), X_train, y_train, cv=cv, scoring=scoring
    )
    rows_no_ord[name] = {
        "RMSE": -res["test_rmse"].mean(),
        "MAE": -res["test_mae"].mean(),
        "R2": res["test_r2"].mean(),
        "RMSLE": -res["test_rmsle"].mean(),
        "RMSLE_std": res["test_rmsle"].std(),
    }

bang_cv_no_ord = pd.DataFrame(rows_no_ord).T.sort_values("RMSLE")
bang_cv_no_ord.round(4)

,RMSE,MAE,R2,RMSLE,RMSLE_std
Lasso,19657.0396,13252.2232,0.9321,0.1097,0.0146
Ridge,19991.6383,13438.7299,0.9296,0.1125,0.0124
HistGB,23176.3827,15189.3460,0.9049,0.1230,0.0153
RandomForest,25247.5955,16608.3548,0.8874,0.1335,0.0133
KNN,31975.6104,20258.3210,0.8192,0.1569,0.0098
DecisionTree,35840.6502,24833.1457,0.7738,0.1968,0.0109
Dummy,76722.8524,54266.3627,-0.0340,0.3887,0.0057


**Nhận xét:**
- Khi tắt `OrdinalMapper`, các cột chất lượng đi qua One-Hot: Lasso đạt RMSLE = 0.1097, Ridge 0.1125, HistGB 0.1230, RandomForest 0.1335 (so với Ordinal: 0.1106, 0.1138, 0.1231, 0.1330).
- One-Hot nhỉnh hơn một chút ở các mô hình tuyến tính và KNN (Lasso giảm 0.0009, Ridge giảm 0.0013, KNN 0.1585 → 0.1569); với HistGB và RandomForest hai cách gần như bằng nhau.
- Mức chênh này nhỏ hơn nhiều so với độ lệch chuẩn RMSLE giữa các fold (khoảng 0.012–0.016), nên **chưa thể kết luận cách mã hóa nào tốt hơn**.
- Nhóm giữ Ordinal Encoding làm mặc định vì bảo toàn trật tự tự nhiên (Po < Fa < TA < Gd < Ex), tạo ít cột hơn và dễ diễn giải; có thể đưa lựa chọn này vào bước tuning (Mục 4) để kiểm tra lại.

## 4. Tinh chỉnh siêu tham số (Hyperparameter Tuning)

> **Phân công nhóm:** Mục này do bạn **Ngô Quang Anh** tiếp tục triển khai (GridSearchCV / RandomizedSearchCV với tiền tố `regressor__model__*` trên mô hình Lasso / Ridge) sau khi merge nhánh `hinh` vào `main`.

## 5. Đánh giá cuối cùng trên tập kiểm tra độc lập (Test Set)

> **Phân công nhóm:** Mục này do bạn **Ngô Quang Anh** thực hiện đánh giá một lần duy nhất cho mô hình tối ưu cuối cùng trên `X_test, y_test` (sử dụng hàm `src.plots.plot_model_evaluation` để xuất biểu đồ 4 góc).